# SQL Worksheet — Week5

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Student Attendance Performance

For every student, calculate total attendance records, Present count, Late count, Absent count, and attendance rate where Present + Late counts as attended. Include students with no attendance records.

In [0]:
select 
    ds.student_id,
    ds.student_name,
    count(fa.attendance_id) as total_records,
    sum(case when fa.attendance_status = 'Present' then 1 else 0 end) as present_count,
    sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) as absent_count,
    sum(case when fa.attendance_status = 'Late' then 1 else 0 end) as late_count,
    (sum(case when fa.attendance_status IN ('Present','Late') then 1 else 0 end) * 100)/
    nullif(count(fa.attendance_id),0) AS attendance_rate

FROM rivadataplatform.dataproduct.dim_student AS ds 
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.student_key = ds.student_key

GROUP BY ds.student_id,ds.student_name
ORDER BY ds.student_id

## Question 2 — Rank Students by Attendance Rate

Using the result from Question 1, return the top 5 students by attendance rate. If two students have the same rate, use `student_name` as the secondary sort.

In [0]:
WITH student_info AS (select 
    ds.student_id,
    ds.student_name,
    count(fa.attendance_id) as total_records,
    sum(case when fa.attendance_status = 'Present' then 1 else 0 end) as present_count,
    sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) as absent_count,
    sum(case when fa.attendance_status = 'Late' then 1 else 0 end) as late_count,
    (sum(case when fa.attendance_status IN ('Present','Late') then 1 else 0 end) * 100)/
    nullif(count(fa.attendance_id),0) AS attendance_rate

FROM rivadataplatform.dataproduct.dim_student AS ds 
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.student_key = ds.student_key

GROUP BY ds.student_id,ds.student_name
ORDER BY ds.student_id )

SELECT 
    student_id,
    student_name,
    attendance_rate
FROM student_info
ORDER BY attendance_rate DESC, student_name ASC
LIMIT 5

## Question 3 — Class Attendance Analysis

For every class, calculate total records, Present count, Late count, Absent count, and attendance rate. Return only classes with at least one attendance record and order by attendance rate descending.

In [0]:
Select 
    dc.class_key,
    count(dc.class_key) As total_records,
    sum(case when fa.attendance_status = 'Present' then 1 else 0 end) AS Present_count,
    sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) AS Absent_count,
    sum(case when fa.attendance_status = 'Late' then 1 else 0 end) AS Late_count,
    sum(case when fa.attendance_status IN ('Present','Late') then 1 else 0 end) AS attendance_record,
    (sum(case when fa.attendance_status IN ('Present','Late') then 1 else 0 end)*100 )/nullif(count(fa.attendance_id),0) AS attendance_rate
FROM rivadataplatform.dataproduct.dim_class AS dc
INNER JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON dc.class_key = fa.class_key
GROUP BY dc.class_key
HAVING attendance_record >= 1
ORDER BY attendance_rate DESC

## Question 4 — Identify High-Risk Students

Find students who have both at least one `Absent` record and at least one `Late` record. Return `student_id`, `student_name`, `absent_count`, and `late_count`.

In [0]:
SELECT 
    ds.student_id,
    ds.student_name,
    sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) as absent_count,
    sum(case when fa.attendance_status = 'Late' then 1 else 0 end) as late_count
FROM rivadataplatform.dataproduct.fact_attendance AS fa
LEFT JOIN rivadataplatform.dataproduct.dim_student AS ds
    ON fa.student_key = ds.student_key
GROUP BY ds.student_id, ds.student_name
HAVING sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) >= 1 AND sum(case when fa.attendance_status = 'Late' then 1 else 0 end) >= 1
ORDER BY ds.student_id

## Question 5 — Attendance Threshold

Find students whose attendance rate is below 80%, considering both Present and Late as attended. Include only students with at least 2 attendance records. Return `student_id`, `student_name`, `attendance_records`, and `attendance_rate`.

In [0]:
SELECT 
    ds.student_id,
    ds.student_name,
    sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) AS attended_records,
    (sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) * 100.0) / 
    nullif(count(fa.attendance_id), 0) AS attendance_rate
FROM rivadataplatform.dataproduct.dim_student AS ds
INNER JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON ds.student_key = fa.student_key
GROUP BY ds.student_id, ds.student_name
HAVING sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) >= 2
ORDER BY ds.student_id

## Question 6 — Multi-table Summary

Build a batch-level attendance summary by joining `dim_batch`, `dim_class`, `fact_attendance`, and `dim_student`. Return batch name, total classes represented in attendance, distinct students represented, total attendance records, Present count, Late count, Absent count, and overall attendance rate. Use aggregation carefully so that student/class rows are not accidentally double-counted.

In [0]:
SELECT 
    db.batch_name,
    count(DISTINCT dc.class_key) AS total_classes,
    count(DISTINCT ds.student_key) AS distinct_students,
    count(DISTINCT fa.attendance_id) AS total_attendance_records,
    sum(case when fa.attendance_status = 'Present' then 1 else 0 end) AS Present_count,
    sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) AS Absent_count,
    sum(case when fa.attendance_status = 'Late' then 1 else 0 end) AS Late_count,
    (sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) * 100.0) / 
    nullif(count(fa.attendance_id), 0) AS attendance_rate
FROM rivadataplatform.dataproduct.dim_batch AS db
LEFT JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON db.batch_id = dc.batch_id
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.class_key = dc.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_student AS ds
    ON ds.student_key = fa.student_key
GROUP BY db.batch_name
ORDER BY attendance_rate DESC NULLs LAST